# Building an Aligned Multi-Tool Research Agent
## Part 1: Mathematical Foundations of RL Alignment

**Learning Objectives:**
- Understand the mathematical foundations of RL alignment from module3.md
- Implement state space architecture for alignment-complete representations
- Build action space with value separability and safety preservation
- Create stochastic reward matrix with problem-dependent distributions

**What You'll Build:**
The core mathematical components that enable aligned decision-making: state representation, action space design, and stochastic reward modeling.

**Prerequisites:**
- Understanding of basic reinforcement learning concepts
- Familiarity with module3.md mathematical framework
- Python programming experience

## Setup and Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
from enum import Enum
import random
from collections import defaultdict
import json
import time
from datetime import datetime

# Optional Claude API - only import if available
try:
    import anthropic
    CLAUDE_AVAILABLE = True
except ImportError:
    CLAUDE_AVAILABLE = False
    print("Claude API not available - running in simulation mode only")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Configure plotting
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')  # Fallback for older versions

sns.set_palette("husl")

print("Dependencies loaded successfully!")
print(f"Claude API available: {CLAUDE_AVAILABLE}")

## Part 1: State Space Architecture

Following module3.md mathematical framework:
$$s = [s_{\text{problem}}, s_{\text{context}}, s_{\text{history}}, s_{\text{resources}}, s_{\text{constraints}}]$$

In [ ]:
@dataclass
class ProblemState:
    """Problem State: s_problem = [query_type, complexity_level, domain, stakeholders]"""
    query_type: str  # 'factual', 'analytical', 'controversial', 'urgent'
    complexity_level: float  # 0.0 to 1.0
    domain: str  # 'science', 'politics', 'technology', 'general'
    stakeholders: List[str]  # affected parties
    
    def to_vector(self) -> np.ndarray:
        """Convert to numerical representation for RL algorithms"""
        query_encoding = {'factual': 0, 'analytical': 1, 'controversial': 2, 'urgent': 3}
        domain_encoding = {'science': 0, 'politics': 1, 'technology': 2, 'general': 3}
        
        return np.array([
            query_encoding.get(self.query_type, 0),
            self.complexity_level,
            domain_encoding.get(self.domain, 0),
            len(self.stakeholders)
        ])

@dataclass 
class ContextState:
    """Context State: s_context = [time_pressure, quality_requirements, user_expertise, urgency_level]"""
    time_pressure: float  # 0.0 to 1.0
    quality_requirements: float  # 0.0 to 1.0  
    user_expertise: float  # 0.0 to 1.0
    urgency_level: float  # 0.0 to 1.0
    
    def to_vector(self) -> np.ndarray:
        return np.array([self.time_pressure, self.quality_requirements, 
                        self.user_expertise, self.urgency_level])

@dataclass
class ResourceState:
    """Resource State: s_resources = [budget_remaining, time_remaining, tool_availability, api_limits]"""
    budget_remaining: float  # 0.0 to 1.0 (normalized)
    time_remaining: float   # 0.0 to 1.0 (normalized)
    tool_availability: Dict[str, bool]  # which tools are available
    api_limits: Dict[str, float]  # remaining API calls per tool
    
    def to_vector(self) -> np.ndarray:
        available_tools = sum(self.tool_availability.values())
        avg_api_remaining = np.mean(list(self.api_limits.values()))
        return np.array([self.budget_remaining, self.time_remaining, 
                        available_tools/12, avg_api_remaining])

@dataclass
class ConstraintState:
    """Constraint State: s_constraints = [privacy_level, compliance_requirements, user_values, safety_thresholds]"""
    privacy_level: float  # 0.0 to 1.0
    compliance_requirements: List[str]  # regulatory constraints
    user_values: Dict[str, float]  # accuracy, speed, cost, safety weights
    safety_thresholds: Dict[str, float]  # minimum safety requirements
    
    def to_vector(self) -> np.ndarray:
        compliance_score = len(self.compliance_requirements) / 5  # normalize
        values_vector = [self.user_values.get(k, 0.5) for k in ['accuracy', 'speed', 'cost', 'safety']]
        safety_score = np.mean(list(self.safety_thresholds.values()))
        return np.array([self.privacy_level, compliance_score, *values_vector, safety_score])

class AgentState:
    """Complete agent state combining all components"""
    def __init__(self, problem: ProblemState, context: ContextState, 
                 resources: ResourceState, constraints: ConstraintState,
                 history: List[Tuple] = None):
        self.problem = problem
        self.context = context 
        self.resources = resources
        self.constraints = constraints
        self.history = history or []  # [(action, reward, outcome), ...]
    
    def to_vector(self) -> np.ndarray:
        """Convert complete state to vector for RL algorithms"""
        # History encoding: last 3 actions and their outcomes
        history_vector = np.zeros(9)  # 3 * (action_id + reward + outcome_quality)
        for i, (action_id, reward, outcome_quality) in enumerate(self.history[-3:]):
            base_idx = i * 3
            history_vector[base_idx:base_idx+3] = [action_id, reward, outcome_quality]
            
        return np.concatenate([
            self.problem.to_vector(),
            self.context.to_vector(), 
            self.resources.to_vector(),
            self.constraints.to_vector(),
            history_vector
        ])
    
    def is_alignment_complete(self) -> bool:
        """Check if state contains sufficient information for aligned decisions"""
        # Implement alignment completeness check from module3.md
        has_user_values = len(self.constraints.user_values) >= 4
        has_safety_thresholds = len(self.constraints.safety_thresholds) >= 2
        has_context = self.context.quality_requirements > 0
        
        return has_user_values and has_safety_thresholds and has_context

# Test the state space
problem = ProblemState('analytical', 0.7, 'science', ['researchers', 'public'])
context = ContextState(0.3, 0.9, 0.6, 0.4)
resources = ResourceState(0.8, 0.7, {f'tool_{i}': True for i in range(12)}, 
                         {f'tool_{i}': 0.9 for i in range(12)})
constraints = ConstraintState(0.5, ['GDPR'], 
                             {'accuracy': 0.8, 'speed': 0.4, 'cost': 0.6, 'safety': 0.9},
                             {'bias': 0.1, 'harm': 0.05})

state = AgentState(problem, context, resources, constraints)
state_vector = state.to_vector()

print(f"State vector dimension: {len(state_vector)}")
print(f"Alignment complete: {state.is_alignment_complete()}")
print(f"Sample state vector: {state_vector[:10]}...")  # First 10 elements

## Part 2: Multi-Tool Action Space

Implementing the 12-tool action space with mathematical properties for alignment:
$$A = \{a_{\text{academic}}, a_{\text{web}}, a_{\text{news}}, a_{\text{fact}}, a_{\text{sentiment}}, a_{\text{cite}}, a_{\text{summarize}}, a_{\text{cross}}, a_{\text{bias}}, a_{\text{confidence}}, a_{\text{human}}, a_{\text{synthesis}}\}$$

In [ ]:
class ResearchTool(Enum):
    """Enumeration of 12 research tools with unique properties"""
    ACADEMIC_SEARCH = 0
    WEB_SEARCH = 1
    NEWS_SEARCH = 2
    FACT_CHECK = 3
    SENTIMENT_ANALYSIS = 4
    CITATION_ANALYSIS = 5
    SUMMARIZATION = 6
    CROSS_REFERENCE = 7
    BIAS_DETECTION = 8
    CONFIDENCE_ASSESSMENT = 9
    HUMAN_CONSULTATION = 10
    SYNTHESIS = 11

@dataclass
class ToolProperties:
    """Tool feature vector: f(a_i) = [c_i, t_i, p_i, s_i]"""
    cost: float          # computational cost
    time: float          # time requirement
    reliability: float   # reliability score
    accuracy: float      # accuracy strength
    speed: float         # speed strength
    coverage: float      # information coverage
    safety: float        # safety level
    
    def feature_vector(self) -> np.ndarray:
        return np.array([self.cost, self.time, self.reliability, 
                        self.accuracy, self.speed, self.coverage, self.safety])

class MultiToolActionSpace:
    """Complete action space with mathematical properties for alignment"""
    
    def __init__(self):
        # Define tool properties based on module3.md specifications
        self.tool_properties = {
            ResearchTool.ACADEMIC_SEARCH: ToolProperties(
                cost=0.7, time=0.8, reliability=0.95, accuracy=0.95, speed=0.2, coverage=0.6, safety=0.9
            ),
            ResearchTool.WEB_SEARCH: ToolProperties(
                cost=0.2, time=0.1, reliability=0.6, accuracy=0.6, speed=0.95, coverage=0.9, safety=0.5
            ),
            ResearchTool.NEWS_SEARCH: ToolProperties(
                cost=0.3, time=0.2, reliability=0.7, accuracy=0.7, speed=0.8, coverage=0.7, safety=0.6
            ),
            ResearchTool.FACT_CHECK: ToolProperties(
                cost=0.8, time=0.6, reliability=0.9, accuracy=0.9, speed=0.4, coverage=0.3, safety=0.95
            ),
            ResearchTool.SENTIMENT_ANALYSIS: ToolProperties(
                cost=0.4, time=0.3, reliability=0.8, accuracy=0.8, speed=0.7, coverage=0.4, safety=0.8
            ),
            ResearchTool.CITATION_ANALYSIS: ToolProperties(
                cost=0.6, time=0.5, reliability=0.85, accuracy=0.85, speed=0.5, coverage=0.5, safety=0.9
            ),
            ResearchTool.SUMMARIZATION: ToolProperties(
                cost=0.3, time=0.2, reliability=0.75, accuracy=0.75, speed=0.8, coverage=0.8, safety=0.7
            ),
            ResearchTool.CROSS_REFERENCE: ToolProperties(
                cost=0.9, time=0.7, reliability=0.9, accuracy=0.9, speed=0.3, coverage=0.9, safety=0.85
            ),
            ResearchTool.BIAS_DETECTION: ToolProperties(
                cost=0.7, time=0.6, reliability=0.85, accuracy=0.85, speed=0.4, coverage=0.3, safety=0.95
            ),
            ResearchTool.CONFIDENCE_ASSESSMENT: ToolProperties(
                cost=0.5, time=0.4, reliability=0.8, accuracy=0.8, speed=0.6, coverage=0.5, safety=0.9
            ),
            ResearchTool.HUMAN_CONSULTATION: ToolProperties(
                cost=1.0, time=1.0, reliability=0.95, accuracy=0.9, speed=0.1, coverage=0.6, safety=1.0
            ),
            ResearchTool.SYNTHESIS: ToolProperties(
                cost=0.8, time=0.9, reliability=0.8, accuracy=0.85, speed=0.2, coverage=0.95, safety=0.8
            )
        }
    
    def get_action_count(self) -> int:
        return len(ResearchTool)
    
    def get_tool_properties(self, tool: ResearchTool) -> ToolProperties:
        return self.tool_properties[tool]
    
    def check_value_separability(self) -> bool:
        """Check if action space satisfies value separability property"""
        # Find tools that maximize each value
        max_accuracy = max(self.tool_properties.values(), key=lambda x: x.accuracy)
        max_speed = max(self.tool_properties.values(), key=lambda x: x.speed)
        max_safety = max(self.tool_properties.values(), key=lambda x: x.safety)
        
        # Check that different tools maximize different values
        separable = (max_accuracy != max_speed or max_accuracy != max_safety or max_speed != max_safety)
        return separable
    
    def get_trade_off_spectrum(self, value1: str, value2: str) -> List[Tuple[ResearchTool, float, float]]:
        """Get tools representing trade-offs between two values"""
        spectrum = []
        for tool, props in self.tool_properties.items():
            v1_score = getattr(props, value1)
            v2_score = getattr(props, value2)
            spectrum.append((tool, v1_score, v2_score))
        
        # Sort by trade-off ratio
        spectrum.sort(key=lambda x: x[1] / (x[2] + 1e-6))
        return spectrum
    
    def has_safety_preservation(self, min_safety: float = 0.8) -> bool:
        """Check if there exists at least one safe action"""
        safe_tools = [tool for tool, props in self.tool_properties.items() 
                     if props.safety >= min_safety]
        return len(safe_tools) > 0
    
    def visualize_action_space(self):
        """Visualize tool properties and trade-offs"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Tool properties heatmap
        tools = list(self.tool_properties.keys())
        properties = ['accuracy', 'speed', 'safety', 'coverage', 'cost', 'time', 'reliability']
        
        data = np.array([[getattr(self.tool_properties[tool], prop) for prop in properties] 
                        for tool in tools])
        
        sns.heatmap(data, xticklabels=properties, 
                   yticklabels=[tool.name.replace('_', ' ') for tool in tools],
                   annot=True, fmt='.2f', ax=axes[0,0])
        axes[0,0].set_title('Tool Properties Matrix')
        
        # 2. Accuracy vs Speed trade-off
        accuracies = [props.accuracy for props in self.tool_properties.values()]
        speeds = [props.speed for props in self.tool_properties.values()]
        tool_names = [tool.name.replace('_', ' ') for tool in tools]
        
        axes[0,1].scatter(speeds, accuracies, s=100, alpha=0.7)
        for i, name in enumerate(tool_names):
            axes[0,1].annotate(name[:8], (speeds[i], accuracies[i]), 
                              xytext=(5,5), textcoords='offset points', fontsize=8)
        axes[0,1].set_xlabel('Speed')
        axes[0,1].set_ylabel('Accuracy')
        axes[0,1].set_title('Accuracy vs Speed Trade-off')
        
        # 3. Cost vs Quality analysis
        costs = [props.cost for props in self.tool_properties.values()]
        qualities = [props.reliability for props in self.tool_properties.values()]
        
        axes[1,0].scatter(costs, qualities, s=100, alpha=0.7, c='green')
        for i, name in enumerate(tool_names):
            axes[1,0].annotate(name[:8], (costs[i], qualities[i]), 
                              xytext=(5,5), textcoords='offset points', fontsize=8)
        axes[1,0].set_xlabel('Cost')
        axes[1,0].set_ylabel('Reliability')
        axes[1,0].set_title('Cost vs Reliability Trade-off')
        
        # 4. Safety scores
        safety_scores = [props.safety for props in self.tool_properties.values()]
        axes[1,1].bar(range(len(tools)), safety_scores, alpha=0.7, color='red')
        axes[1,1].set_xticks(range(len(tools)))
        axes[1,1].set_xticklabels([tool.name.replace('_', ' ')[:8] for tool in tools], 
                                 rotation=45)
        axes[1,1].set_ylabel('Safety Score')
        axes[1,1].set_title('Safety Scores by Tool')
        axes[1,1].axhline(y=0.8, color='red', linestyle='--', label='Min Safety Threshold')
        axes[1,1].legend()
        
        plt.tight_layout()
        plt.show()

# Test the action space
action_space = MultiToolActionSpace()

print(f"Action space size: {action_space.get_action_count()}")
print(f"Value separability: {action_space.check_value_separability()}")
print(f"Safety preservation: {action_space.has_safety_preservation()}")

# Show accuracy vs speed trade-off spectrum
trade_offs = action_space.get_trade_off_spectrum('accuracy', 'speed')
print("\nAccuracy vs Speed Trade-off Spectrum:")
for tool, acc, speed in trade_offs[:5]:
    print(f"{tool.name}: Accuracy={acc:.2f}, Speed={speed:.2f}")

In [ ]:
# Visualize the action space
action_space.visualize_action_space()

## Part 3: Stochastic Reward Matrix

Implementing the multi-dimensional stochastic reward structure:
$$R(s,a,c) = R_{\text{base}}(s,a) + R_{\text{context}}(c) + R_{\text{alignment}}(s,a,v) + \epsilon$$

In [ ]:
class StochasticRewardMatrix:
    """Implements stochastic reward distributions for the multi-tool agent"""
    
    def __init__(self, action_space: MultiToolActionSpace):
        self.action_space = action_space
        
        # Problem-type dependent distributions (from module3.md)
        self.problem_distributions = {
            'controversial': {
                ResearchTool.ACADEMIC_SEARCH: {'mean': 8.0, 'std': 1.5},
                ResearchTool.WEB_SEARCH: {'mean': 3.0, 'std': 2.5},
                ResearchTool.FACT_CHECK: {'mean': 8.5, 'std': 1.2},
                ResearchTool.BIAS_DETECTION: {'mean': 9.0, 'std': 1.0},
                ResearchTool.HUMAN_CONSULTATION: {'mean': 8.8, 'std': 0.8}
            },
            'urgent': {
                ResearchTool.ACADEMIC_SEARCH: {'mean': 4.0, 'std': 2.0},
                ResearchTool.WEB_SEARCH: {'mean': 7.5, 'std': 1.8},
                ResearchTool.NEWS_SEARCH: {'mean': 8.0, 'std': 1.5},
                ResearchTool.SUMMARIZATION: {'mean': 7.0, 'std': 1.6}
            },
            'factual': {
                ResearchTool.ACADEMIC_SEARCH: {'mean': 9.0, 'std': 1.0},
                ResearchTool.FACT_CHECK: {'mean': 9.2, 'std': 0.8},
                ResearchTool.CITATION_ANALYSIS: {'mean': 8.5, 'std': 1.2},
                ResearchTool.CROSS_REFERENCE: {'mean': 8.8, 'std': 1.1}
            },
            'analytical': {
                ResearchTool.SYNTHESIS: {'mean': 8.5, 'std': 1.3},
                ResearchTool.CROSS_REFERENCE: {'mean': 8.0, 'std': 1.4},
                ResearchTool.CONFIDENCE_ASSESSMENT: {'mean': 7.8, 'std': 1.5},
                ResearchTool.ACADEMIC_SEARCH: {'mean': 8.2, 'std': 1.2}
            }
        }
        
        # Default distributions for tools not specified
        self.default_distribution = {'mean': 5.0, 'std': 2.0}
    
    def get_base_reward(self, state: AgentState, action: ResearchTool) -> float:
        """R_base(s,a): Base effectiveness of tool a in state s"""
        problem_type = state.problem.query_type
        
        # Get problem-specific distribution or default
        if problem_type in self.problem_distributions:
            if action in self.problem_distributions[problem_type]:
                dist = self.problem_distributions[problem_type][action]
            else:
                dist = self.default_distribution
        else:
            dist = self.default_distribution
        
        # Sample from normal distribution
        reward = np.random.normal(dist['mean'], dist['std'])
        
        # Apply complexity modifier
        complexity_modifier = 1.0 - (state.problem.complexity_level * 0.3)
        
        return max(0, reward * complexity_modifier)
    
    def get_context_reward(self, state: AgentState, action: ResearchTool) -> float:
        """R_context(c): Contextual modifiers based on time pressure, resources, etc."""
        tool_props = self.action_space.get_tool_properties(action)
        
        # Time pressure penalty for slow tools
        time_penalty = state.context.time_pressure * tool_props.time * 2.0
        
        # Quality bonus for high-quality requirements and reliable tools
        quality_bonus = state.context.quality_requirements * tool_props.reliability * 1.5
        
        # Resource penalty for expensive tools when resources are low
        resource_penalty = (1 - state.resources.budget_remaining) * tool_props.cost * 1.5
        
        # Urgency modifier
        if state.context.urgency_level > 0.7 and tool_props.speed < 0.5:
            urgency_penalty = 2.0
        else:
            urgency_penalty = 0.0
        
        return quality_bonus - time_penalty - resource_penalty - urgency_penalty
    
    def get_alignment_reward(self, state: AgentState, action: ResearchTool) -> float:
        """R_alignment(s,a,v): Alignment with user values"""
        tool_props = self.action_space.get_tool_properties(action)
        user_values = state.constraints.user_values
        
        # Calculate value-weighted score
        alignment_score = (
            user_values.get('accuracy', 0.5) * tool_props.accuracy +
            user_values.get('speed', 0.5) * tool_props.speed +
            user_values.get('cost', 0.5) * (1 - tool_props.cost) +  # Lower cost = higher score
            user_values.get('safety', 0.5) * tool_props.safety
        )
        
        # Safety constraint violation penalty
        safety_penalty = 0.0
        for constraint, threshold in state.constraints.safety_thresholds.items():
            if constraint == 'bias' and tool_props.safety < (1 - threshold):
                safety_penalty += 3.0
            elif constraint == 'harm' and tool_props.safety < (1 - threshold):
                safety_penalty += 5.0
        
        return alignment_score * 2.0 - safety_penalty
    
    def get_stochastic_reward(self, state: AgentState, action: ResearchTool) -> float:
        """Complete stochastic reward function: R(s,a,c) = R_base + R_context + R_alignment + ε"""
        base_reward = self.get_base_reward(state, action)
        context_reward = self.get_context_reward(state, action)
        alignment_reward = self.get_alignment_reward(state, action)
        
        # Random noise: ε ~ N(0, σ²)
        noise = np.random.normal(0, 0.5)
        
        total_reward = base_reward + context_reward + alignment_reward + noise
        
        return total_reward
    
    def analyze_reward_distributions(self, num_samples: int = 1000):
        """Analyze reward distributions across different scenarios"""
        scenarios = {
            'controversial_high_quality': AgentState(
                ProblemState('controversial', 0.8, 'politics', ['public', 'media']),
                ContextState(0.3, 0.9, 0.6, 0.5),
                ResourceState(0.8, 0.7, {tool.name: True for tool in ResearchTool}, 
                             {tool.name: 0.9 for tool in ResearchTool}),
                ConstraintState(0.7, ['ethical'], 
                               {'accuracy': 0.9, 'speed': 0.3, 'cost': 0.5, 'safety': 0.95},
                               {'bias': 0.1, 'harm': 0.05})
            ),
            'urgent_time_pressure': AgentState(
                ProblemState('urgent', 0.4, 'technology', ['stakeholders']),
                ContextState(0.9, 0.6, 0.7, 0.9),
                ResourceState(0.3, 0.2, {tool.name: True for tool in ResearchTool}, 
                             {tool.name: 0.4 for tool in ResearchTool}),
                ConstraintState(0.4, [], 
                               {'accuracy': 0.6, 'speed': 0.9, 'cost': 0.8, 'safety': 0.7},
                               {'bias': 0.2, 'harm': 0.1})
            )
        }
        
        results = {}
        
        for scenario_name, state in scenarios.items():
            results[scenario_name] = {}
            
            for tool in ResearchTool:
                rewards = [self.get_stochastic_reward(state, tool) for _ in range(num_samples)]
                results[scenario_name][tool] = {
                    'mean': np.mean(rewards),
                    'std': np.std(rewards),
                    'samples': rewards
                }
        
        return results
    
    def visualize_reward_distributions(self, analysis_results: dict):
        """Visualize reward distributions across scenarios and tools"""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        scenarios = list(analysis_results.keys())
        tools = list(ResearchTool)
        
        # 1. Mean rewards heatmap
        mean_data = np.array([[analysis_results[scenario][tool]['mean'] 
                              for tool in tools] for scenario in scenarios])
        
        sns.heatmap(mean_data, 
                   xticklabels=[tool.name.replace('_', ' ')[:10] for tool in tools],
                   yticklabels=scenarios,
                   annot=True, fmt='.1f', ax=axes[0,0])
        axes[0,0].set_title('Mean Rewards by Scenario and Tool')
        axes[0,0].set_xlabel('Tools')
        axes[0,0].set_ylabel('Scenarios')
        
        # 2. Standard deviation heatmap
        std_data = np.array([[analysis_results[scenario][tool]['std'] 
                             for tool in tools] for scenario in scenarios])
        
        sns.heatmap(std_data, 
                   xticklabels=[tool.name.replace('_', ' ')[:10] for tool in tools],
                   yticklabels=scenarios,
                   annot=True, fmt='.2f', ax=axes[0,1], cmap='YlOrRd')
        axes[0,1].set_title('Reward Standard Deviation')
        axes[0,1].set_xlabel('Tools')
        axes[0,1].set_ylabel('Scenarios')
        
        # 3. Reward distributions for controversial scenario
        controversial_tools = [ResearchTool.ACADEMIC_SEARCH, ResearchTool.WEB_SEARCH, 
                              ResearchTool.FACT_CHECK, ResearchTool.BIAS_DETECTION]
        
        for i, tool in enumerate(controversial_tools):
            samples = analysis_results[scenarios[0]][tool]['samples']
            axes[1,0].hist(samples, alpha=0.6, label=tool.name.replace('_', ' ')[:10], bins=20)
        
        axes[1,0].set_title('Reward Distributions - Controversial Scenario')
        axes[1,0].set_xlabel('Reward Value')
        axes[1,0].set_ylabel('Frequency')
        axes[1,0].legend()
        
        # 4. Risk-return analysis
        for scenario in scenarios:
            means = [analysis_results[scenario][tool]['mean'] for tool in tools]
            stds = [analysis_results[scenario][tool]['std'] for tool in tools]
            axes[1,1].scatter(stds, means, label=scenario, alpha=0.7, s=60)
        
        axes[1,1].set_xlabel('Risk (Standard Deviation)')
        axes[1,1].set_ylabel('Expected Return (Mean)')
        axes[1,1].set_title('Risk-Return Analysis by Scenario')
        axes[1,1].legend()
        
        plt.tight_layout()
        plt.show()

# Test the stochastic reward matrix
reward_matrix = StochasticRewardMatrix(action_space)

# Create test state
test_state = AgentState(
    ProblemState('controversial', 0.7, 'politics', ['public']),
    ContextState(0.4, 0.8, 0.6, 0.6),
    ResourceState(0.7, 0.6, {tool.name: True for tool in ResearchTool}, 
                 {tool.name: 0.8 for tool in ResearchTool}),
    ConstraintState(0.6, ['GDPR'], 
                   {'accuracy': 0.8, 'speed': 0.4, 'cost': 0.6, 'safety': 0.9},
                   {'bias': 0.1, 'harm': 0.05})
)

# Test reward components for academic search
action = ResearchTool.ACADEMIC_SEARCH
base_r = reward_matrix.get_base_reward(test_state, action)
context_r = reward_matrix.get_context_reward(test_state, action)
alignment_r = reward_matrix.get_alignment_reward(test_state, action)
total_r = reward_matrix.get_stochastic_reward(test_state, action)

print(f"Reward Components for {action.name}:")
print(f"Base: {base_r:.2f}")
print(f"Context: {context_r:.2f}")
print(f"Alignment: {alignment_r:.2f}")
print(f"Total: {total_r:.2f}")

# Analyze distributions
print("\nAnalyzing reward distributions...")
analysis = reward_matrix.analyze_reward_distributions(500)

# Show top tools for each scenario
for scenario, tools_data in analysis.items():
    sorted_tools = sorted(tools_data.items(), key=lambda x: x[1]['mean'], reverse=True)
    print(f"\nTop 3 tools for {scenario}:")
    for tool, data in sorted_tools[:3]:
        print(f"  {tool.name}: μ={data['mean']:.2f}, σ={data['std']:.2f}")

In [ ]:
# Visualize reward distributions
reward_matrix.visualize_reward_distributions(analysis)

## Key Learning Outcomes - Part 1

You've successfully implemented the mathematical foundations of RL alignment:

### 1. **State Space Architecture**
- ✅ **Alignment-Complete Representation**: State includes user values, constraints, and context
- ✅ **Multi-Dimensional Structure**: Problem, context, resources, constraints, and history
- ✅ **Mathematical Encoding**: Proper vectorization for RL algorithms

### 2. **Action Space Design**
- ✅ **Value Separability**: Different tools maximize different values (accuracy vs speed vs safety)
- ✅ **Trade-off Continuity**: Spectrum of tools representing different value compromises
- ✅ **Safety Preservation**: Always exists at least one safe action option

### 3. **Stochastic Reward Structure**
- ✅ **Multi-Component Rewards**: Base effectiveness + Context + Alignment + Noise
- ✅ **Problem-Dependent Distributions**: Same tool performs differently in different contexts
- ✅ **Uncertainty Modeling**: Stochastic rewards force robust policy learning

## Next Steps

**Continue to Part 2** to learn how these mathematical foundations enable:
- Trajectory-level alignment constraints
- Progressive curriculum learning
- Transfer learning between complexity stages

**Experiment Ideas:**
1. Modify user value weights and observe tool selection changes
2. Test different problem types and complexity levels
3. Analyze reward variance across different scenarios
4. Design your own research tools with specific properties

The mathematical foundations you've built here will directly enable the sophisticated learning and alignment behaviors in the next notebooks!